In [ ]:
# Shared setup and reusable notebook helpers
import importlib
import os
import subprocess
import sys
from pathlib import Path
import numpy as np
import illustris_python as il

root = Path("/public/home/zju_visitor/LiuYuanhao/SAM_project/bymyself")
data_dir = root / "data"
basePath = "/public/share/chenhouzun/TNG50-1/output/"
snaps = [99, 67, 40]

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

env = os.environ.copy()
env["PYTHONPATH"] = str(root) + os.pathsep + env.get("PYTHONPATH", "")

def run_script(script, *args):
    subprocess.run([sys.executable, str(script), *map(str, args)], cwd=root, env=env, check=True)

def run_snapshots(script, output_template, label, snap_list=snaps):
    for snap in snap_list:
        output_file = data_dir / output_template.format(snap=snap)
        if output_file.is_file():
            print(f"{label} file for snap {snap} already exists, skip.")
            continue
        print(f"Running {label} calculation for snap {snap}")
        run_script(script, snap)
    print(f"All {label} calculations finished.")

def run_plot(script, finished_message):
    run_script(script)
    print(finished_message)


In [ ]:
# Cooling-radius calculation and mass-trend plotting
cooling_script = root / "cooling_radius.py"
plot_script = root / "plotting_scripts" / "plot_cooling_radius_mass_trend.py"
run_snapshots(cooling_script, "cooling_radius_{snap}.hdf5", "Cooling")
run_plot(plot_script, "Plotting finished.")

In [ ]:
# sSFR classification and plotting
ssfr_script = root / "ssfr_classify_galaxies.py"
ssfr_plot_script = root / "plotting_scripts" / "plot_cooling_radius_ssfr.py"
run_snapshots(ssfr_script, "ssfr_classification_{snap}.hdf5", "sSFR classification")
run_plot(ssfr_plot_script, "sSFR plotting finished.")


In [ ]:
# Run the standalone sSFR–M200c plotting script for snaps [99, 67, 40]
ssfr_m200c_plot_script = root / "plotting_scripts" / "plot_ssfr_m200c_classification.py"
run_plot(ssfr_m200c_plot_script, "sSFR versus M200c plots finished.")


In [ ]:
# AGN f_Edd classification and plotting
agn_script = root / "agn_classify_galaxies.py"
agn_plot_script = root / "plotting_scripts" / "plot_cooling_radius_agn.py"
run_snapshots(agn_script, "fedd_classification_{snap}.hdf5", "AGN classification")
run_plot(agn_plot_script, "AGN plotting finished.")


In [ ]:
# Cold-gas fraction calculation and Rcool phase-diagram plotting_1
cold_fraction_script = root / "cold_gas_fraction.py"
cold_fraction_plot_script = root / "plotting_scripts" / "plot_cold_gas_fraction_R_cool_trend_1.py"
run_snapshots(cold_fraction_script, "cold_gas_fraction_{snap}_1.hdf5", "Cold-gas fraction")
run_plot(cold_fraction_plot_script, "Cold-gas fraction plotting finished.")


In [ ]:
# Cold-gas fraction calculation and Rcool phase-diagram plotting
cold_fraction_script = root / "cold_gas_fraction.py"
cold_fraction_plot_script = root / "plotting_scripts" / "plot_cold_gas_fraction_R_cool_trend.py"
run_snapshots(cold_fraction_script, "cold_gas_fraction_{snap}.hdf5", "Cold-gas fraction")
run_plot(cold_fraction_plot_script, "Cold-gas fraction plotting finished.")


In [ ]:
# Position histories, selected subhalo sample, and always-central trees
import create_position_histories as cph
import create_subhalo_sample as css
importlib.reload(cph)
importlib.reload(css)

start_snap = 99
snap_min = 40
snap_max = 99
position_file = data_dir / "SubhaloPos_new.hdf5"
extrapolated_file = data_dir / "SubhaloPos_new_extrapolated.hdf5"
tree_file = data_dir / "all_selected_trees.hdf5"
data_dir.mkdir(parents=True, exist_ok=True)

if tree_file.is_file():
    print(f"Selected-tree file already exists, skip: {tree_file}")
else:
    if not position_file.is_file():
        print("Creating subhalo position histories.")
        cph.save_subhalo_pos(basePath)
    if not extrapolated_file.is_file():
        position_snaps = np.arange(snap_max, snap_min - 1, -1, dtype=np.int64)
        headers = [il.groupcat.loadHeader(basePath, int(snap)) for snap in position_snaps]
        box_size = headers[0]["BoxSize"]
        scale_factors = 1.0 / (1.0 + np.asarray([header["Redshift"] for header in headers]))
        cph.extrapolatePos(basePath, box_size, scale_factors, 0.1)
    subhaloFlag = css.create_subhalo_sample(basePath, start_snap)
    selected_start_ids, selected_node_ids, selected_fof_ids = cph.save_always_central_trees(basePath, start_snap, subhaloFlag, snap_min=snap_min, snap_max=snap_max)
    print("Selected subhalos:", np.count_nonzero(subhaloFlag))
    print("Always-central trees:", selected_start_ids.size)
    print("Tree node array shape:", selected_node_ids.shape)
    print("Tree FoF array shape:", selected_fof_ids.shape)


In [ ]:
# Gas parent classification: select inner star-forming or cold gas cells
tree_file = data_dir / "all_selected_trees.hdf5"
data_dir.mkdir(parents=True, exist_ok=True)

if not tree_file.is_file():
    raise FileNotFoundError(f"Selected-tree file does not exist: {tree_file}. Run the selected-tree creation step first.")

import gas_parent_classification as gpc
importlib.reload(gpc)  # load the latest saved version of the module

pending_snaps = [snap for snap in snaps if not (data_dir / f"gas_parent_classification_{snap}.hdf5").is_file()]

if pending_snaps:
    print("Running gas-parent classification for snapshots:", pending_snaps)
    for snap in pending_snaps:
        gpc.gas_parent_classification(basePath, snap)
    print("Gas-parent classification finished.")
else:
    print("Gas-parent classification files already exist; skipped all snapshots.")


In [ ]:
# Classify start-snapshot gas and trace its parents to the target snapshots
import gas_parent_classification as gpc
import tracerFuncs as tF
importlib.reload(gpc)
importlib.reload(tF)

tracer_jobs = [(95, target_snap) for target_snap in range(90, 95)]
tracer_jobs.extend((94, target_snap) for target_snap in range(95, 100))

required_inputs = [data_dir / "all_selected_trees.hdf5"]
missing_inputs = [path for path in required_inputs if not path.is_file()]
if missing_inputs:
    raise FileNotFoundError("Missing inputs required by gas-parent classification:\n" + "\n".join(map(str, missing_inputs)))

for tracer_start_snap in sorted({start_snap for start_snap, _ in tracer_jobs}):
    gas_class_file = data_dir / f"gas_parent_classification_{tracer_start_snap}.hdf5"
    if gas_class_file.is_file():
        print(f"Gas-parent classification already exists, reuse: {gas_class_file}")
    else:
        print(f"Classifying start-snapshot gas for snap {tracer_start_snap}", flush=True)
        gpc.gas_parent_classification(basePath, tracer_start_snap)

for tracer_start_snap, tracer_target_snap in tracer_jobs:
    parent_file = data_dir / f"parent_indices_start_{tracer_start_snap}_target_{tracer_target_snap}.hdf5"
    if parent_file.is_file():
        print(f"Tracer-parent file already exists, skip: {parent_file}")
    else:
        print(f"Tracing inner-gas parents: start_snap={tracer_start_snap}, target_snap={tracer_target_snap}", flush=True)
        tF.TraceBackAllInnerGas(basePath, tracer_start_snap, tracer_target_snap)

print("Requested tracer-parent files are ready.")


In [ ]:
# Plot tracer parent-type fractions versus stellar mass for the traced snapshot ranges
tracer_fraction_plot_script = root / "plotting_scripts" / "plot_tracer_fraction_stellar_mass_trend.py"
tracer_fraction_jobs = [(95, list(range(90, 95))), (94, list(range(95, 100)))]

for tracer_start_snap, tracer_target_snaps in tracer_fraction_jobs:
    required_files = [data_dir / "all_selected_trees.hdf5", data_dir / f"gas_parent_classification_{tracer_start_snap}.hdf5"]
    required_files.extend(data_dir / f"parent_indices_start_{tracer_start_snap}_target_{target_snap}.hdf5" for target_snap in tracer_target_snaps)
    missing_files = [path for path in required_files if not path.is_file()]
    if missing_files:
        raise FileNotFoundError("Missing inputs required by the tracer-fraction plot:\n" + "\n".join(map(str, missing_files)))
    print(f"Plotting tracer fractions: start_snap={tracer_start_snap}, target_snaps={tracer_target_snaps}", flush=True)
    run_script(tracer_fraction_plot_script, 1, tracer_start_snap, *tracer_target_snaps)

print("Tracer-fraction stellar-mass plots finished.")

In [ ]:
# Classify tracer distance and temperature for the requested snapshot pairs
import distance_temp_cats as dtc
importlib.reload(dtc)

distance_jobs = [(95, target_snap) for target_snap in range(90, 95)]
distance_jobs.extend((94, target_snap) for target_snap in range(95, 100))
distance_output_dir = data_dir / "distance_cats"
distance_output_dir.mkdir(parents=True, exist_ok=True)

pending_jobs = []
for distance_start_snap, distance_target_snap in distance_jobs:
    output_file = distance_output_dir / f"distance_cats_start_{distance_start_snap}_target_{distance_target_snap}.hdf5"
    if output_file.is_file():
        print(f"Distance-temperature file already exists, skip: {output_file}")
        continue
    pending_jobs.append((distance_start_snap, distance_target_snap))

if pending_jobs:
    required_inputs = [data_dir / "all_selected_trees.hdf5", data_dir / "SubhaloPos_new_extrapolated.hdf5", data_dir / "subhaloFlag.hdf5"]
    required_inputs.extend(data_dir / f"gas_parent_classification_{start_snap}.hdf5" for start_snap in sorted({start_snap for start_snap, _ in pending_jobs}))
    required_inputs.extend(data_dir / f"parent_indices_start_{start_snap}_target_{target_snap}.hdf5" for start_snap, target_snap in pending_jobs)
    missing_inputs = [path for path in required_inputs if not path.is_file()]
    if missing_inputs:
        raise FileNotFoundError("Missing inputs required by distance_temp_cats:\n" + "\n".join(map(str, missing_inputs)))

for distance_start_snap, distance_target_snap in pending_jobs:
    print(f"Running distance-temperature classification: start_snap={distance_start_snap}, target_snap={distance_target_snap}", flush=True)
    dtc.distance_temp_cats(basePath, distance_start_snap, distance_target_snap)

print("Requested distance-temperature classifications finished.")


In [ ]:
# sSFR-classified cold-gas fraction and Rcool phase diagrams
plot_script = root / "plotting_scripts" / "plot_cold_gas_fraction_R_cool_trend_ssfr.py"
required_files = [path for snap in snaps for path in (data_dir / f"cooling_radius_{snap}.hdf5", data_dir / f"cold_gas_fraction_{snap}.hdf5", data_dir / f"ssfr_classification_{snap}.hdf5")]
run_plot(plot_script, "sSFR-classified cold-gas phase diagrams finished.")


In [ ]:
# Plot distance-temperature class fractions versus stellar mass for the requested snapshot pairs
distance_fraction_plot_script = root / "plotting_scripts" / "plot_accretion_channels_mass_trend.py"
distance_fraction_plot_jobs = [(94, [97, 99]), (95, [92, 90])]

for plot_start_snap, plot_target_snaps in distance_fraction_plot_jobs:
    required_files = [data_dir / "all_selected_trees.hdf5"]
    required_files.extend(data_dir / "distance_cats" / f"distance_cats_start_{plot_start_snap}_target_{target_snap}.hdf5" for target_snap in plot_target_snaps)
    missing_files = [path for path in required_files if not path.is_file()]
    if missing_files:
        raise FileNotFoundError("Missing inputs required by the distance-fraction plot:\n" + "\n".join(map(str, missing_files)))
    print(f"Plotting distance fractions: start_snap={plot_start_snap}, target_snaps={plot_target_snaps}", flush=True)
    run_script(distance_fraction_plot_script, plot_start_snap, *plot_target_snaps)

print("Requested distance-fraction plots finished.")


In [ ]:
# Lgal cooling rate between snaps 94 and 95
import cooling_rate_funcs as crf
importlib.reload(crf)
crf.Lgal_cooling_rate(basePath, snap_1=94, snap_2=95)


In [ ]:
# Tracer cooling rate: start at snap 95, classify at snap 94
import cooling_rate_funcs as crf
importlib.reload(crf)
crf.tracer_cooling_rate_direct(basePath, target_snap=94, start_snap=95)


In [ ]:
# Tracer out rate: start at snap 94, classify at snap 95
import cooling_rate_funcs as crf
importlib.reload(crf)
crf.tracer_out_rate_direct(basePath, target_snap=95, start_snap=94)


In [ ]:
# Prepare sSFR classifications for snaps 94 and 95, then calculate the modified tracer cooling rate
import cooling_radius as cr
import cold_gas_fraction as cgf
import ssfr_classify_galaxies as scg
import cooling_rate_funcs as crf
importlib.reload(cr)
importlib.reload(cgf)
importlib.reload(scg)
importlib.reload(crf)

for ssfr_snap in [94, 95]:
    ssfr_file = data_dir / f"ssfr_classification_{ssfr_snap}.hdf5"
    if ssfr_file.is_file():
        print(f"sSFR classification already exists, reuse: {ssfr_file}")
        continue

    cooling_file = data_dir / f"cooling_radius_{ssfr_snap}.hdf5"
    if not cooling_file.is_file():
        print(f"Calculating cooling radius for snap {ssfr_snap}", flush=True)
        cr.cooling_radius(basePath, ssfr_snap)

    cold_fraction_file = data_dir / f"cold_gas_fraction_{ssfr_snap}_1.hdf5"
    if not cold_fraction_file.is_file():
        print(f"Calculating cold-gas fraction for snap {ssfr_snap}", flush=True)
        cgf.cold_gas_fraction(basePath, ssfr_snap)

    print(f"Calculating sSFR classification for snap {ssfr_snap}", flush=True)
    scg.ssfr_classification(basePath, ssfr_snap)

crf.tracer_cooling_rate_modified(basePath, target_snaps_forward=[90, 91, 92, 93, 94], start_snap_backward=94, start_snap_forward=95, target_snaps_backward=[95, 96, 97, 98, 99])


In [ ]:
# Plot cooling-rate relations for snaps 94 and 95
cooling_rate_plot_script = root / "plotting_scripts" / "plot_cooling_rates.py"
cooling_rate_plot_inputs = [
    data_dir / "all_selected_trees.hdf5",
    data_dir / "Lgal_cooling_rate_94_and_95.hdf5",
    data_dir / "distance_cats" / "tcrd_94_to_95.hdf5",
    data_dir / "distance_cats" / "tord_94_to_95.hdf5",
    data_dir / "distance_cats" / "tracer_cooling_rate_modified_94_to_95.hdf5",
]
missing_cooling_rate_inputs = [path for path in cooling_rate_plot_inputs if not path.is_file()]
if missing_cooling_rate_inputs:
    raise FileNotFoundError("Missing inputs required by plot_cooling_rates.py:\n" + "\n".join(map(str, missing_cooling_rate_inputs)))

run_plot(cooling_rate_plot_script, "Cooling-rate plots finished.")


In [ ]:
# Plot modified tracer fractions versus the corresponding snapshot stellar mass
tracer_fraction_modified_plot_script = root / "plotting_scripts" / "plot_tracer_fraction_stellar_mass_trend_1.py"
tracer_fraction_modified_inputs = [
    data_dir / "all_selected_trees.hdf5",
    data_dir / "distance_cats" / "tracer_cooling_rate_modified_94_to_95.hdf5",
]
missing_tracer_fraction_modified_inputs = [path for path in tracer_fraction_modified_inputs if not path.is_file()]
if missing_tracer_fraction_modified_inputs:
    raise FileNotFoundError("Missing inputs required by plot_tracer_fraction_stellar_mass_trend_1.py:\n" + "\n".join(map(str, missing_tracer_fraction_modified_inputs)))

run_plot(tracer_fraction_modified_plot_script, "Modified tracer-fraction stellar-mass plot finished.")


In [ ]:
# Plot cooling-radius PDFs for snaps 40, 67, and 99
cooling_radius_pdf_script = root / "plotting_scripts" / "plot_pdfs_cooling_radius.py"
cooling_radius_pdf_inputs = [data_dir / f"cooling_radius_{snap}.hdf5" for snap in [40, 67, 99]]
missing_cooling_radius_pdf_inputs = [path for path in cooling_radius_pdf_inputs if not path.is_file()]
if missing_cooling_radius_pdf_inputs:
    raise FileNotFoundError("Missing cooling-radius PDF inputs:\n" + "\n".join(map(str, missing_cooling_radius_pdf_inputs)))

run_plot(cooling_radius_pdf_script, "Cooling-radius PDF plot finished.")


In [ ]:
# Plot cold-gas-fraction PDFs by cooling-radius interval for snaps 40, 67, and 99
cold_gas_pdf_script = root / "plotting_scripts" / "plot_pdfs_cold_gas.py"
cold_gas_pdf_inputs = [data_dir / f"cooling_radius_{snap}.hdf5" for snap in [40, 67, 99]]
cold_gas_pdf_inputs.extend(data_dir / f"cold_gas_fraction_{snap}_1.hdf5" for snap in [40, 67, 99])
missing_cold_gas_pdf_inputs = [path for path in cold_gas_pdf_inputs if not path.is_file()]
if missing_cold_gas_pdf_inputs:
    raise FileNotFoundError("Missing inputs required by plot_pdfs_cold_gas.py:\n" + "\n".join(map(str, missing_cold_gas_pdf_inputs)))

run_plot(cold_gas_pdf_script, "Cold-gas-fraction PDF plot finished.")


In [ ]:
# Calculate median profiles for all selected halos in the three mass ranges
import real_physical_properties as rpp
importlib.reload(rpp)
properties_snap = 99
properties_result = rpp.real_physical_properties(basePath, snap=properties_snap)


In [ ]:
# Plot one combined median-profile figure for all selected halos at the chosen snapshot
properties_plot_script = root / "plotting_scripts" / "plot_properties_tng_Lgal.py"
properties_snap = 99
properties_file = data_dir / f"real_physical_properties_{properties_snap}.hdf5"
if not properties_file.is_file():
    raise FileNotFoundError(f"Run the radial-profile calculation first: {properties_file}")
run_script(properties_plot_script, properties_snap)


In [ ]:
# Calculate radial physical properties for all selected merger-tree halos at snaps 99, 67, and 40
import real_physical_properties_all as rppa
importlib.reload(rppa)

real_properties_snaps = [99, 67, 40]
for properties_snap in real_properties_snaps:
    real_properties_file = data_dir / f"real_physical_properties_all_{properties_snap}.hdf5"
    if real_properties_file.is_file():
        print(f"Real physical-properties file already exists, reuse: {real_properties_file}")
    else:
        print(f"Calculating real physical properties for all selected halos at snap {properties_snap}", flush=True)
        rppa.real_physical_properties(basePath, snap=properties_snap)


In [ ]:
# Calculate real-profile cooling radii from the measured profiles at snaps 99, 67, and 40
import real_cooling_radius as rcr
importlib.reload(rcr)

real_cooling_snaps = [99, 67, 40]
for cooling_snap in real_cooling_snaps:
    real_properties_file = data_dir / f"real_physical_properties_all_{cooling_snap}.hdf5"
    if not real_properties_file.is_file():
        raise FileNotFoundError(f"Run the all-halo physical-properties calculation first: {real_properties_file}")

    real_cooling_file = data_dir / f"real_cooling_radius_{cooling_snap}.hdf5"
    if real_cooling_file.is_file():
        print(f"Real cooling-radius file already exists, reuse: {real_cooling_file}")
    else:
        print(f"Calculating real-profile cooling radius for snap {cooling_snap}", flush=True)
        rcr.real_cooling_radius(cooling_snap, properties_file=real_properties_file)

# Keep the following comparison-plot cell configured for z=0.
real_cooling_snap = 99


In [ ]:
# Compare analytic and TNG measured-profile cooling radii at z=0
cooling_radius_comparison_script = root / "plotting_scripts" / "plot_Lgal_tng_cooling_radius_mass_trend.py"
cooling_radius_comparison_inputs = [
    data_dir / "all_selected_trees.hdf5",
    data_dir / f"cooling_radius_{real_cooling_snap}.hdf5",
    data_dir / f"real_cooling_radius_{real_cooling_snap}.hdf5",
]
missing_cooling_radius_comparison_inputs = [path for path in cooling_radius_comparison_inputs if not path.is_file()]
if missing_cooling_radius_comparison_inputs:
    raise FileNotFoundError("Missing inputs required by the cooling-radius comparison plot:\n" + "\n".join(map(str, missing_cooling_radius_comparison_inputs)))

run_plot(cooling_radius_comparison_script, "L-GALAXIES versus TNG cooling-radius plot finished.")


In [ ]:
# Plot analytic and TNG measured-profile cooling radii for snaps 99, 67, and 40
cooling_radius_redshift_plot_script = root / "plotting_scripts" / "plot_Lgal_tng_cooling_radius_mass_trend_z.py"
cooling_radius_redshift_plot_inputs = [data_dir / "all_selected_trees.hdf5"]
cooling_radius_redshift_plot_inputs.extend(data_dir / f"cooling_radius_{snap}.hdf5" for snap in [99, 67, 40])
cooling_radius_redshift_plot_inputs.extend(data_dir / f"real_cooling_radius_{snap}.hdf5" for snap in [99, 67, 40])
missing_cooling_radius_redshift_inputs = [path for path in cooling_radius_redshift_plot_inputs if not path.is_file()]
if missing_cooling_radius_redshift_inputs:
    raise FileNotFoundError("Missing inputs required by the redshift cooling-radius plot:\n" + "\n".join(map(str, missing_cooling_radius_redshift_inputs)))

run_plot(cooling_radius_redshift_plot_script, "Multi-redshift L-GALAXIES versus TNG cooling-radius plot finished.")
